# 09 — NumPy และ Pandas เบื้องต้น

**คอร์ส Python for Geography** · ภาควิชาภูมิศาสตร์ ชั้นปีที่ 2
**วันที่ 5 · ช่วงที่ 2 · เวลาโดยประมาณ 60 นาที**

---

### สิ่งที่จะได้จากบทนี้
- สร้างและคำนวณกับ **NumPy array** แบบ vectorized (ไม่ต้องเขียนลูป)
- ใช้ **boolean masking** และจัดการค่า `NaN`
- อ่าน CSV เข้า **pandas DataFrame** แล้วเลือก กรอง จัดกลุ่ม เรียงลำดับ
- ทำงานกับวันที่ และสรุปข้อมูลรายเดือน/รายสถานี

> **วิธีเรียน:** โค้ดในเซลล์สีเทาคือโค้ดที่อาจารย์จะรันให้ดู
> ใต้แต่ละหัวข้อจะมีเซลล์ ✍️ ว่างไว้ — **ให้พิมพ์ตามด้วยตัวเอง ห้าม copy–paste**
> การพิมพ์เองคือวิธีเดียวที่จะจำ syntax ได้

## 0. เตรียมไฟล์ข้อมูล

In [ ]:
# ===== เตรียมไฟล์ข้อมูลสำหรับบทนี้ (รันครั้งเดียวต่อ session) =====
# ถ้าอาจารย์เปลี่ยนชื่อ repo ให้แก้ URL บรรทัดล่างนี้บรรทัดเดียว
BASE = "https://raw.githubusercontent.com/jamorn12/Teaching_Python_for_Geography/main/"

import os, subprocess, sys, urllib.request

FILES = ["stations.csv", "rainfall_daily_2025.csv", "messy_rainfall.csv"]
os.makedirs("data", exist_ok=True)


def have_all():
    return all(os.path.exists(os.path.join("data", n)) for n in FILES)


# 1) ถ้ามีไฟล์อยู่แล้ว (เช่น clone repo มา หรืออัปโหลดเอง) ก็ใช้ได้เลย
if not have_all():
    # 2) ลองดาวน์โหลดจาก GitHub
    for name in FILES:
        target = os.path.join("data", name)
        if not os.path.exists(target):
            try:
                urllib.request.urlretrieve(BASE + "data/" + name, target)
            except Exception:
                pass

if not have_all():
    # 3) สำรองสุดท้าย: สร้างข้อมูลชุดเดียวกันขึ้นมาใหม่จาก make_datasets.py
    script = None
    for cand in ["make_datasets.py", "utils/make_datasets.py"]:
        if os.path.exists(cand):
            script = cand
            break
    if script is None:
        try:
            urllib.request.urlretrieve(BASE + "utils/make_datasets.py", "make_datasets.py")
            script = "make_datasets.py"
        except Exception:
            script = None
    if script:
        subprocess.run([sys.executable, script, "--outdir", "data"], check=True)

if have_all():
    print("ข้อมูลพร้อมใช้งาน:", sorted(os.listdir("data")))
else:
    print("ยังไม่พบข้อมูล -> ให้ upload โฟลเดอร์ data/ เข้ามา "
          "หรือรัน utils/make_datasets.py เพื่อสร้างข้อมูลใหม่")

## 1. NumPy — คำนวณทั้งชุดข้อมูลพร้อมกัน

`ndarray` ต่างจาก list ตรงที่เก็บข้อมูลชนิดเดียว และคำนวณทั้งก้อนได้ในคำสั่งเดียว (**vectorization**)
เร็วกว่าลูปหลายสิบเท่า และเป็นพื้นฐานของ pandas, xarray, rasterio ที่ใช้ในงาน GIS/อุตุนิยมวิทยา

In [ ]:
import numpy as np

rain = np.array([0.0, 12.4, 0.0, 45.5, 118.6, 3.2, 0.0, 88.9])

print(rain)
print("shape:", rain.shape, "| dtype:", rain.dtype, "| ndim:", rain.ndim)

print(rain / 25.4)          # แปลงเป็นนิ้ว ทั้งชุด ไม่ต้องมีลูป
print(rain * 2)
print(rain + 1)

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

In [ ]:
print("sum   :", rain.sum())
print("mean  :", rain.mean().round(2))
print("max   :", rain.max(), "ที่ตำแหน่ง", rain.argmax())
print("min   :", rain.min())
print("std   :", rain.std().round(2))

print(np.arange(0, 10, 2))
print(np.linspace(0, 1, 5))
print(np.zeros(4), np.ones(3))

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### Boolean masking — กรองข้อมูลด้วยเงื่อนไข

หัวใจของการวิเคราะห์ข้อมูล: สร้าง array ของ `True`/`False` แล้วใช้เลือกค่า

In [ ]:
mask = rain > 35
print(mask)
print(rain[mask])
print("จำนวนวันฝนหนัก:", mask.sum())        # True นับเป็น 1
print("วันที่ (index):", np.where(mask)[0])

# np.where ใช้แทน if ทั้งชุดได้
labels = np.where(rain >= 0.1, "ฝนตก", "ไม่มีฝน")
print(labels)

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### `NaN` — ค่าที่หายไป

ในข้อมูลอุตุนิยมวิทยา ค่าที่วัดไม่ได้จะแทนด้วย `np.nan`
ฟังก์ชันปกติจะได้ `nan` ทั้งหมด ต้องใช้เวอร์ชัน `nan...` แทน

In [ ]:
x = np.array([12.4, np.nan, 45.5, 118.6, np.nan])

print(x.mean())            # nan
print(np.nanmean(x))       # ข้ามค่า nan
print(np.nansum(x))
print(np.isnan(x))
print("จำนวนค่าที่หาย:", np.isnan(x).sum())
print("เอาเฉพาะค่าที่ใช้ได้:", x[~np.isnan(x)])   # ~ คือ not

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### Array 2 มิติ — ต้นแบบของ raster grid

In [ ]:
grid = np.array([
    [0.0,  5.2, 12.1],
    [3.4, 18.9, 25.0],
    [1.1,  8.7, 40.2],
])

print(grid.shape)
print(grid[0, 2], grid[-1, -1])
print(grid[:, 1])            # คอลัมน์กลาง
print(grid.sum())
print("รวมตามแถว   :", grid.sum(axis=1))
print("รวมตามคอลัมน์:", grid.sum(axis=0))

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 2. Pandas — ตารางข้อมูล

`DataFrame` = ตารางที่มีชื่อคอลัมน์ คิดง่าย ๆ ว่าเป็น Excel ที่สั่งงานด้วยโค้ดได้

In [ ]:
import pandas as pd

df = pd.read_csv("data/rainfall_daily_2025.csv")

print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print(df.columns.tolist())
print(df.dtypes)
df.describe()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### เลือกข้อมูล: คอลัมน์, `loc`, `iloc`, และการกรอง

In [ ]:
print(df["rain_mm"].head())          # 1 คอลัมน์ -> Series
df[["date", "station_id", "rain_mm"]].head()

In [ ]:
print(df.iloc[0])            # แถวที่ 0 ตามตำแหน่ง
print(df.iloc[0:3, 0:3])     # แถว 0-2 คอลัมน์ 0-2
print(df.loc[0, "rain_mm"])  # ตามชื่อ label

In [ ]:
# กรองด้วยเงื่อนไข (boolean masking เหมือน numpy)
chb = df[df["station_id"] == "CHB01"]
print(chb.shape)

heavy = df[df["rain_mm"] > 90]
print(heavy[["date", "station_id", "rain_mm"]])

# หลายเงื่อนไข ต้องใส่วงเล็บทุกอัน และใช้ & | แทน and or
sep_heavy = df[(df["rain_mm"] > 35) & (df["date"].str.startswith("2025-09"))]
print(sep_heavy.shape)

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### วันที่ และคอลัมน์ใหม่

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month
df["rain_inch"] = (df["rain_mm"] / 25.4).round(3)
df["is_rain"] = df["rain_mm"] >= 0.1

df.head()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### `groupby()` — สรุปแบบแบ่งกลุ่ม

นี่คือคำสั่งที่ทรงพลังที่สุดใน pandas: **แบ่งกลุ่ม → คำนวณ → รวมผล**

In [ ]:
# ฝนรวมรายสถานี
print(df.groupby("station_id")["rain_mm"].sum().round(1))

# ฝนรวมรายเดือนของแต่ละสถานี
monthly = df.groupby(["station_id", "month"])["rain_mm"].sum().round(1)
print(monthly.head(15))

# หลายสถิติพร้อมกัน
summary = df.groupby("station_id").agg(
    total_mm=("rain_mm", "sum"),
    max_daily=("rain_mm", "max"),
    mean_daily=("rain_mm", "mean"),
    rain_days=("is_rain", "sum"),
).round(2)
summary

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

In [ ]:
# เรียงลำดับ และดู 5 วันที่ฝนมากที่สุดของทั้งชุดข้อมูล
top5 = df.sort_values("rain_mm", ascending=False).head(5)
top5[["date", "station_id", "station_name", "rain_mm"]]

In [ ]:
# รวมตารางสองตาราง (merge) — เหมือน join ในฐานข้อมูล/GIS attribute table
stations = pd.read_csv("data/stations.csv")
merged = summary.reset_index().merge(stations, on="station_id")
merged[["station_id", "station_name", "lat", "lon", "total_mm", "rain_days"]]

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### บันทึกผลลัพธ์

In [ ]:
merged.to_csv("annual_summary_2025.csv", index=False, encoding="utf-8-sig")
!head -3 annual_summary_2025.csv

## 3. แบบฝึกหัดท้ายบท

1. หาว่าเดือนไหนของสถานี `NYI01` มีฝนรวมมากที่สุด และมากเท่าไร
2. นับจำนวนวันที่ฝน > 35 mm ของแต่ละสถานี
3. สร้างคอลัมน์ `level` ที่จำแนกระดับฝน แล้วใช้ `value_counts()` นับจำนวนวันแต่ละระดับ
   (ใบ้: ใช้ `pd.cut()` หรือ `.apply()` กับฟังก์ชันที่เขียนไว้บทที่ 07)
4. หา "ฝนสะสม 3 วัน" สูงสุดของสถานี `CHB01` (ใบ้: `.rolling(3).sum()`)

In [ ]:
# ข้อ 1

In [ ]:
# ข้อ 2

In [ ]:
# ข้อ 3

In [ ]:
# ข้อ 4

### เฉลย
```python
# ข้อ 1
nyi = df[df["station_id"] == "NYI01"].groupby("month")["rain_mm"].sum().round(1)
print(nyi)
print("เดือนฝนมากสุด:", nyi.idxmax(), "=", nyi.max(), "mm")

# ข้อ 2
print(df[df["rain_mm"] > 35].groupby("station_id").size())

# ข้อ 3
def classify_rain(mm):
    if mm < 0.1:   return "ไม่มีฝน"
    if mm <= 10:   return "ฝนเล็กน้อย"
    if mm <= 35:   return "ฝนปานกลาง"
    if mm <= 90:   return "ฝนหนัก"
    return "ฝนหนักมาก"

df["level"] = df["rain_mm"].apply(classify_rain)
print(df["level"].value_counts())
print(pd.crosstab(df["station_id"], df["level"]))

# ข้อ 4
chb = df[df["station_id"] == "CHB01"].sort_values("date").copy()
chb["rain_3d"] = chb["rain_mm"].rolling(3).sum()
row = chb.loc[chb["rain_3d"].idxmax()]
print(f"ฝนสะสม 3 วันสูงสุด {row['rain_3d']:.1f} mm สิ้นสุดวันที่ {row['date'].date()}")
```